In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
#Stock has been sampled at a 20 minutes frequency here
N_per_h = 3

Metropolis_params = {
    "eval_hours": [N_per_h* h for h in [5, 11, 16, 21]],
    "levels": (
        {"empty": 0.22, "full": 0.66}
    )
}

#shortcuts
Metro_h = Metropolis_params["eval_hours"]
Metro_l = Metropolis_params["levels"]

def Metropolis_metric(stock, on_Metro_h=False):
    if on_Metro_h:
        select_h = Metro_h
    else:
        select_h = slice(None) #if stock is already processeed to match metro_h hours

    mean_s = stock.mean(axis =1) #/!\ mean over days

    mean_min = mean_s[:, select_h].min(axis=1) #/!\ min over hour
    mean_max = mean_s[:, select_h].max(axis=1) #/!\ max over hour

    test = {"empty": (mean_max < Metro_l["empty"]),
            "full": (mean_min > Metro_l["full"])}
    return test


In [ ]:
dir = "envelop_inputs/"
n_stations = 10
sim_envelops = {}

select_h = [0] + Metro_h + [-1]

inspect_shape = True
for d_name in["xmin_out", "xmax_out", "xmin_in", "xmax_in"]:
    envlp = np.load(dir + d_name + ".npy").astype(int)
    envlp_select = envlp[:, :, select_h]
    sim_envelops[d_name] = envlp_select
    if inspect_shape:
        print("selected times per day for simulation", select_h)
        dims = envlp.shape
        print("enevelops input shape (stations, days, times per day)", dims)
        dims_sel = envlp_select.shape
        print("selected shape (stations, days, times per day)", dims_sel)
        inspect_shape = False

capacity = np.max(sim_envelops["xmax_out"], axis = (1, 2))
n_stations = len(capacity)

selected times per day for simulation [0, 15, 33, 48, 63, -1]
enevelops input shape (stations, days, times per day) (10, 7, 73)
selected shape (stations, days, times per day) (10, 7, 6)


In [ ]:
#we recall last (rebalancing free) stock updates of the bike park at 23h59 (or 22h/23h in practice)
#here random updates
last_updates = [np.random.choice(int(c)) for c in capacity]

In [ ]:
#we apply envelop formula:
#stock(s, t) = PI_{s,t}(stock(s, 0) + delta(s,t))
#where PI: projection on outer envelop
#  y(x) = PI_{s,t}(x) = clip(x, xmin_out(s,t), xmax_out(s,t))

#delta: variation of inner envelop
# delta(s, t) = xmin*_in(s,t) - xmin*_in(s,0)
#  *doesn't matter if we select min or max here, or even half of both
class Simulator:
    def __init__(self, envelops, last_updates, test_metric, truck_tolerance=5):
        self.init_stock = last_updates
        self.envlp = envelops
        self.test_metric = test_metric
        self.truck_tolerance =  truck_tolerance
        self.stock_null = self.simulate_null()

    def simulate_null(self):
        #simulate null policy to allow impact evaluation
        null_policy = np.zeros(7, dtype = int)
        stock_null = self.__call__(null_policy, remove_midnight = {"current": False, "next": True}, output = "stock_only")
        return stock_null


    def proj(self, x, day):
        low_x, up_x = self.envlp["xmin_out"][:, day], self.envlp["xmax_out"][:, day]
        return np.clip(x, low_x, up_x)

    def delta(self, day):
        #here we could use min or max, or even average of both
        value_0 = self.envlp["xmin_in"][:, day, 0:1]
        value_h = self.envlp["xmin_in"][:, day]
        return value_h - value_0

    def stock_dynamic(self, init, day):
        return self.proj(init[:, None] + self.delta(day), day)

    def balance_criterion(self, stock):
        #balance_criterion complete the metropolis metric
        # which is blind to some possible positive impact
        # of rebalancing
        target = 0.5*(self.envlp["xmin_in"] + self.envlp["xmax_in"])
        target_midnight = target[:, :, 0]
        stock_midnight = stock[:, :, 0]
        return np.sum(np.abs(target_midnight-stock_midnight), axis =1)


    def test_buffer(self, stock):
        #buffer station are station were rebalancing might be pertinent,
        #even when the test_metric doesn't reflect that because test metric
        #is incomplete
        balance_null = self.balance_criterion(self.stock_null)
        balance_here = self.balance_criterion(stock)
        criterion = (balance_here <= balance_null +0.1)
        return criterion


    def __call__(self, policy,  remove_midnight={"current": True, "next": True}, output = "all"):
        #===== check truck feasability ======
        feasible = np.full(n_stations, True)
        tol = self.truck_tolerance
        #===== boundary conditions (inter day link)
        add_current_midnight = 1
        add_next_midnight = 1
        remove_current_midnight = 1 if remove_midnight["current"] else 0
        remove_next_midnight = -1 if remove_midnight["next"] else None
        #======= simulating ... =============
        stock = np.zeros((n_stations, 7, add_current_midnight + len(Metro_h) +add_next_midnight)).astype(int)
        end_last_day = self.init_stock
        for day in range(7):
            init_day = end_last_day + policy[day]

            #track truck feasability
            feasible = feasible & (
                (-tol <=init_day) & (init_day <= capacity + tol)
            )
            stock[:, day] = self.stock_dynamic(init_day, day)
            end_last_day = stock[:, day, -1]

        if output == "all":
            test_buffer = self.test_buffer(stock)

        stock = stock[:, :, remove_current_midnight:remove_next_midnight]

        if output == "stock_only":
            return stock

        elif output == "all":
            return {"-": "stock output shape: (station, day, metropolis_hour)",
                    "shape": stock.shape,
                    "stock": stock,
                    "feasible": feasible,
                    "test_metric": self.test_metric(stock),
                    "test_buffer": test_buffer
                    }
        else:
            raise ValueError('output must be "all" or "stock_only"')

In [ ]:
simulate = Simulator(sim_envelops, last_updates, Metropolis_metric)

null_policy = np.zeros(7, dtype = int)
print("simulating null policy:", null_policy)
simulate(null_policy)

simulating null policy: [0 0 0 0 0 0 0]


{'-': 'stock output shape: (station, day, metropolis_hour)',
 'shape': (10, 7, 4),
 'stock': array([[[ 4, 20, 20, 17],
         [10, 20, 20, 16],
         [16, 20, 20, 14],
         [18, 20, 20,  0],
         [ 0, 18, 20, 11],
         [10, 14, 20, 16],
         [15, 13, 13, 13]],
 
        [[ 2,  1,  0,  7],
         [20, 12,  8, 10],
         [19,  0,  0, 20],
         [20,  0,  0, 20],
         [18,  0,  0,  2],
         [17, 15, 19,  8],
         [20, 19, 19, 20]],
 
        [[ 8,  0,  0,  0],
         [ 2,  0,  0,  4],
         [14,  0,  0,  0],
         [ 0,  0,  0,  1],
         [ 0, 13, 10,  6],
         [16, 16,  7,  3],
         [ 5,  8,  0,  2]],
 
        [[ 1,  0,  5,  7],
         [15, 15,  8,  7],
         [19,  7, 13,  8],
         [20,  0,  0,  9],
         [19,  3,  9, 12],
         [20, 16, 13, 13],
         [ 9,  4,  6,  3]],
 
        [[ 2,  2,  2,  2],
         [ 2,  2,  2,  2],
         [ 2,  2,  2,  2],
         [ 2,  2,  2,  2],
         [ 2,  2,  2,  2],
     

In [ ]:
simulate = Simulator(sim_envelops, last_updates, Metropolis_metric)

policy = np.full(7, -15)
print("simulating  massive extraction policy:", policy)
simulate(policy)

simulating  massive extraction policy: [-15 -15 -15 -15 -15 -15 -15]


{'-': 'stock output shape: (station, day, metropolis_hour)',
 'shape': (10, 7, 4),
 'stock': array([[[ 0, 20, 20, 17],
         [ 0, 14, 20, 16],
         [ 1, 20, 20, 14],
         [ 9, 20, 20,  0],
         [ 0, 18, 20, 11],
         [ 0,  6, 20, 16],
         [ 0,  0,  0,  0]],
 
        [[ 2,  1,  0,  7],
         [13, 12,  8, 10],
         [ 5,  0,  0, 20],
         [10,  0,  0, 20],
         [ 5,  0,  0,  2],
         [15, 14, 19,  8],
         [20, 19, 19, 20]],
 
        [[ 0,  0,  0,  0],
         [ 1,  0,  0,  4],
         [ 6,  0,  0,  0],
         [ 0,  0,  0,  1],
         [ 0, 13, 10,  6],
         [ 5,  5,  0,  0],
         [ 2,  6,  0,  2]],
 
        [[ 0,  0,  5,  7],
         [ 4,  4,  0,  1],
         [ 7,  0,  7,  3],
         [10,  0,  0,  9],
         [ 7,  0,  7,  9],
         [16, 16, 13, 13],
         [ 2,  0,  3,  0]],
 
        [[ 0,  0,  0,  0],
         [ 0,  0,  0,  0],
         [ 0,  0,  0,  0],
         [ 0,  0,  0,  0],
         [ 0,  0,  0,  0],
     

In [ ]:
class StationType:
    def __init__(self):
        self.simulate = simulate
        self.ptypes = np.full(n_stations, "------", dtype = "U10")
        self.btypes = np.full((n_stations, 7), "------", dtype = "U10")
        self.classify()

    def classify(self):
        #null policy
        null_policy = np.zeros(7, dtype = int)
        test_metric = simulate(null_policy)["test_metric"]

        for s in range(n_stations):
            if test_metric["empty"][s]:
                self.ptypes[s] = "P-Empt"
            if test_metric["full"][s]:
                self.ptypes[s] = "P-Full"

        #single day policys: check for possible buffers
        problem = ["empt", "full"]
        opposit_problem = ["full", "empty"]
        policy_value = [+15, -15]
        for i in range(2):
            pb, opposit_pb, day_val = problem[i], opposit_problem[i], policy_value[i]
            for day in range(7):
                day_policy = null_policy.copy()
                day_policy[day] = day_val
                sim = simulate(day_policy)
                test_metric_day = sim["test_metric"]
                test_buffer_day = sim["test_buffer"]
                feasible = sim["feasible"]
                for s in range(n_stations):
                    criterion = (feasible[s]
                        & test_buffer_day[s]
                        & (not test_metric_day[opposit_pb][s])
                    )
                    if criterion:
                        self.btypes[s, day] = "B-" + pb

Final output: a class value for each station

In [ ]:
StationTp = StationType()
Ptypes = StationTp.ptypes
Btypes = StationTp.btypes
print("priority stations==============================")
print(StationTp.ptypes)
print("buffer stations and corresponding day==========")
print(StationTp.btypes)

priority stations==============================
['P-Full' 'P-Full' 'P-Full' 'P-Full' 'P-Full' 'P-Full' 'P-Full' '------'
 '------' 'P-Full']
buffer stations and corresponding day==========
[['------' 'B-full' 'B-full' 'B-full' '------' 'B-full' '------']
 ['------' 'B-full' '------' '------' '------' '------' 'B-full']
 ['------' '------' '------' '------' '------' '------' '------']
 ['------' '------' '------' '------' 'B-full' 'B-full' '------']
 ['------' '------' '------' '------' '------' '------' '------']
 ['------' 'B-full' 'B-full' '------' '------' '------' '------']
 ['------' '------' '------' '------' '------' '------' '------']
 ['B-empt' '------' 'B-empt' '------' '------' 'B-empt' '------']
 ['B-empt' 'B-empt' 'B-empt' '------' '------' 'B-full' '------']
 ['B-full' 'B-full' 'B-full' 'B-full' 'B-full' 'B-full' '------']]
